### Блок 1. Уравнение и начальные условия

Решаем волновое уравнение

$$u_{tt}(x,t) = c^2 u_{xx}(x,t), \quad 0 \le x \le L.$$

Используем два варианта начальных условий \(u(x,0)=f(x)\), \(u_t(x,0)=0\)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Параметры задачи
L = 1.0
c = 1.0

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.3
})

def initial_pluck(x, A=1.0):
    """
    f(x) = u(x, 0), g(x) = 0.
    """
    y = np.zeros_like(x)
    m = x <= 0.5 * L
    y[m] = (2 * A / L) * x[m]
    y[~m] = (2 * A / L) * (L - x[~m])
    return y

def initial_mode(x, n=1, A=1.0):
    """
    sin(n*pi*x/L).
    """
    return A * np.sin(n * np.pi * x / L)



Сетка: $x_i = i\Delta x$, $t^n = n\Delta t$,  
$\lambda = \dfrac{c\Delta t}{\Delta x}$ — CFL.

Схема «крест»:

$$
u_i^{n+1} = 2u_i^n - u_i^{n-1}
+ \lambda^2\big(u_{i+1}^n - 2u_i^n + u_{i-1}^n\big).
$$

Слева всегда $u_0^n = 0$.

Справа:
* `fixed`: $u_{N-1}^n = 0$;
* `free`: свободный конец, $u_x(L,t)=0$,  
  тогда $u_{xx}(L,t) \approx 2\,(u_{N-2}^n - u_{N-1}^n)/\Delta x^2$.

Дискретная энергия:

$$
E^n \approx \tfrac12 \sum_i (v_i^n)^2 \Delta x
+ \tfrac12 c^2 \sum_i (d_i^n)^2 \Delta x,
$$

где $v_i^n \approx (u_i^n-u_i^{n-1})/\Delta t$,  
$d_i^n \approx (u_{i+1}^n-u_i^n)/\Delta x$.


In [ ]:
def simulate_wave(
    N,
    CFL,
    T,
    A=1.0,
    init_type="pluck",   # "pluck" или "mode"
    mode_n=1,
    bc_right="fixed",    # "fixed" или "free"
    blowup_threshold=50.0
):

    dx = L / (N - 1)
    dt = CFL * dx / c
    Nt = int(np.floor(T / dt))

    x = np.linspace(0.0, L, N)

    # начальное смещение f(x)
    if init_type == "pluck":
        u0 = initial_pluck(x, A=A)
    elif init_type == "mode":
        u0 = initial_mode(x, n=mode_n, A=A)
    else:
        raise ValueError("init_type должен быть 'pluck' или 'mode'")

    # начальная скорость g(x) = 0
    g = np.zeros_like(x)

    # граничные условия при t = 0
    # левая граница всегда закреплена
    u0[0] = 0.0

    # правая граница
    if bc_right == "fixed":
        u0[-1] = 0.0
    elif bc_right == "free":

        pass
    else:
        raise ValueError("bc_right должен быть 'fixed' или 'free'")

    u1 = u0.copy()
    u1[1:-1] = (
        u0[1:-1]
        + dt * g[1:-1]
        + 0.5 * (CFL**2) * (u0[2:] - 2 * u0[1:-1] + u0[:-2])
    )

    # левая граница (закреплена)
    u1[0] = 0.0

    # правая граница
    if bc_right == "fixed":
        u1[-1] = 0.0
    else:
        lap_last = 2.0 * (u0[-2] - u0[-1])
        u1[-1] = u0[-1] + dt * g[-1] + 0.5 * (CFL**2) * lap_last

    U = [u0.copy(), u1.copy()]
    times = [0.0, dt]
    E = []

    for n in range(1, Nt):
        u_nm1 = U[-2]
        u_n = U[-1]
        u_np1 = np.zeros_like(u_n)

        # внутренние точки
        u_np1[1:-1] = (
            2 * u_n[1:-1]
            - u_nm1[1:-1]
            + (CFL**2) * (u_n[2:] - 2 * u_n[1:-1] + u_n[:-2])
        )

        # левая граница (закреплена)
        u_np1[0] = 0.0

        # правая граница
        if bc_right == "fixed":
            u_np1[-1] = 0.0
        else:
            lap_last_n = 2.0 * (u_n[-2] - u_n[-1])
            u_np1[-1] = 2 * u_n[-1] - u_nm1[-1] + (CFL**2) * lap_last_n

        # дискретная энергия на момент времени t_n
        vel = (u_n - u_nm1) / dt
        du = np.diff(u_n) / dx
        E.append(0.5 * np.sum(vel**2) * dx + 0.5 * (c**2) * np.sum(du**2) * dx)

        U.append(u_np1)
        times.append(times[-1] + dt)

        if np.isnan(u_np1).any() or np.max(np.abs(u_np1)) > blowup_threshold:
            print("Решение улетело при t =", times[-1])
            break

    return x, np.array(U), np.array(times), np.array(E), dt, dx


In [ ]:
def make_animation(x, U, times, A=1.0, fps=30, max_frames=240, title="1D wave"):

    stride = max(1, int(np.ceil(len(times) / max_frames)))
    idx = np.arange(0, len(times), stride)

    fig, ax = plt.subplots()
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$u(x,t)$")
    ax.set_title(title)
    ax.set_xlim(0, L)
    ax.set_ylim(-1.2 * A, 1.2 * A)

    line, = ax.plot(x, U[0], lw=2)
    txt = ax.text(0.02, 0.92, "", transform=ax.transAxes)

    def update(frame):
        k = idx[frame]
        line.set_ydata(U[k])
        txt.set_text(f"t = {times[k]:.3f}")
        return line, txt

    anim = FuncAnimation(fig, update, frames=len(idx), interval=1000 / fps, blit=True)

    try:
        from IPython.display import HTML, display
        display(HTML(anim.to_jshtml()))
        plt.close(fig)
    except Exception:
        plt.show()


def plot_energy(times, E, label=""):

    plt.figure()
    plt.plot(times[1:len(E) + 1], E, lw=2, label=label if label else None)
    plt.xlabel(r"$t$")
    plt.ylabel(r"$E(t)$")
    plt.title("Дискретная энергия E(t)")
    if label:
        plt.legend()
    plt.tight_layout()
    plt.show()


Закреплённые концы и исследование CFL

Граничные условия: $u(0,t)=0$, $u(L,t)=0$.

Начальное условие — «щипок» в центре.

Исследуем разные значения $\lambda = c\Delta t/\Delta x$  
и смотрим на форму решения и на поведение энергии $E^n$
(при $\lambda>1$ схема становится неустойчивой).



In [ ]:
def demo_fixed_fixed():

    N = 201
    T = 4.0
    A = 1.0
    CFLS = [0.20, 0.50, 0.90, 1.00, 1.05]

    energy_sets = []
    time_sets = []

    for CFL in CFLS:
        print(f"\n=== CFL = {CFL:.2f}, закреплённые концы ===")
        x, U, times, E, dt, dx = simulate_wave(
            N=N,
            CFL=CFL,
            T=T,
            A=A,
            init_type="pluck",
            bc_right="fixed"
        )

        make_animation(
            x, U, times, A=A,
            title=f"Струна, закреплённые концы, CFL = {CFL:.2f}"
        )
        plot_energy(times, E, label=f"CFL = {CFL:.2f}")

        energy_sets.append(E)
        time_sets.append(times)

    plt.figure()
    for CFL, times, E in zip(CFLS, time_sets, energy_sets):
        plt.plot(times[1:len(E) + 1], E, lw=2, label=f"CFL = {CFL:.2f}")
    plt.xlabel(r"$t$")
    plt.ylabel(r"$E(t)$")
    plt.title("Сравнение энергии для разных CFL, закреплённые концы")
    plt.legend()
    plt.tight_layout()
    plt.show()


### Свободный конец

Левый конец: $u(0,t)=0$ (закреплён).  
Правый конец свободный: $u_x(L,t)=0$.



In [ ]:
def demo_free_end():

    N = 201
    T = 4.0
    A = 1.0
    CFL = 0.90

    print(f"\n=== CFL = {CFL:.2f}, левая граница закреплена, правая свободна ===")
    x, U, times, E, dt, dx = simulate_wave(
        N=N,
        CFL=CFL,
        T=T,
        A=A,
        init_type="pluck",
        bc_right="free"
    )

    make_animation(
        x, U, times, A=A,
        title=f"Струна: левая закреплена, правая свободна, CFL = {CFL:.2f}"
    )
    plot_energy(times, E, label="Neumann справа (свободный конец)")


### Стоячие волны $$\sin(n\pi x/L)$$

Оба конца закреплены.

Начальное условие для моды $n$:
$$u(x,0) = A\sin\left(\frac{n\pi x}{L}\right), \quad u_t(x,0)=0.$$

форма по $x$ почти неизменна,
а энергия $E^n$ близка к постоянной.


In [ ]:
def demo_standing_wave(n=1):

    N = 201
    T = 4.0
    A = 1.0
    CFL = 0.90

    print(f"\n=== Стоячая волна, мода n = {n}, CFL = {CFL:.2f} ===")
    x, U, times, E, dt, dx = simulate_wave(
        N=N,
        CFL=CFL,
        T=T,
        A=A,
        init_type="mode",
        mode_n=n,
        bc_right="fixed"
    )

    make_animation(
        x, U, times, A=A,
        title=f"Стоячая волна: sin({n}πx/L), CFL = {CFL:.2f}"
    )
    plot_energy(times, E, label=f"Стоячая волна, мода n = {n}")

In [ ]:
demo_fixed_fixed()

In [ ]:
demo_free_end()

In [ ]:
demo_standing_wave(n=1)


In [ ]:
demo_standing_wave(n=2)


In [ ]:
demo_standing_wave(n=3)